In [9]:
import pandas as pd
from pathlib import Path
import calendar

In [10]:
db_path = Path(r"E:\ProyectoAnalisisElectrico\MedidasValorizadas")
to_save_path = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")

In [11]:
def get_days_in_month(date_int):
    year = date_int // 100
    month = date_int % 100
    _, days = calendar.monthrange(2000 + year, month)
    return days

In [12]:
group_columns = [
    'clave',
    'nombre_barra',
    'tension',
    'Zona',
    'Razon_Social',
    'RUT',
    'Nombre_Corto',
    'Hora_Dia',
    'Año_Mes',
    'tipo'
]

In [13]:
agg_rules = {
    'medida_3_sum': ['mean', 'std', 'count'], 
    'CMg[CLP/KWh]_mean': ['mean', 'std', 'count'],
    'valorizado_CLP_sum': ['mean', 'std', 'count']
}

In [ ]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    date_str = folder_date.name
    
    if int(date_str) < 2505: 
        continue

    to_save_folder = to_save_path / f"{date_str}"
    n_days = get_days_in_month(int(date_str))
    
    if to_save_folder.is_dir():
        print(f"Data for {date_str} already processed. Skipping...")
        continue  
        
    to_save_folder.mkdir(parents=True, exist_ok=True)
    parquet_file = folder_date / f"{date_str}_medidas_horarias.parquet"

    print(f"Starting {date_str}...")

    df = pd.read_parquet(parquet_file)  
    
    df["Fecha_Medicion_last"] = pd.to_datetime(df["Fecha_Medicion_last"], format="%Y-%m-%d %H:%M:%S")
    df["Hora_Dia"] = df["Fecha_Medicion_last"].dt.hour
    df["Año_Mes"] = df["Fecha_Medicion_last"].dt.to_period('M').dt.to_timestamp()

    groups = df.groupby(by=group_columns, sort=False) 
    groups_with_size = groups.size()

    valid_groups_mask = (groups_with_size == n_days)
    
    df_aggregated = groups.agg(agg_rules)
    

    df_aggregated.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0] for col in df_aggregated.columns]
    
    df_valid = df_aggregated[valid_groups_mask].reset_index()
    
    rename_dict = {
        'Hora_Dia': 'Hora',
        'medida_3_sum_mean': 'medida_mean',
        'medida_3_sum_std': 'medida_std',
        'medida_3_sum_count': 'medida_count',
        'CMg[CLP/KWh]_mean_mean': 'CMg[CLP/KWh]_mean',
        'CMg[CLP/KWh]_mean_std': 'CMg[CLP/KWh]_std',
        'CMg[CLP/KWh]_mean_count': 'CMg[CLP/KWh]_count',
        'valorizado_CLP_sum_mean': 'valorizado_CLP_mean',
        'valorizado_CLP_sum_std': 'valorizado_CLP_std',
        'valorizado_CLP_sum_count': 'valorizado_CLP_count'
    }
    df_valid = df_valid.rename(columns=rename_dict)
    
    df_valid.to_parquet(to_save_folder / f"{date_str}_mean_month.parquet", engine="pyarrow", compression="snappy")

    invalid_groups_mask = ~valid_groups_mask
    if invalid_groups_mask.any(): 
        print(f"Issues detected in {date_str} with {parquet_file.name}. Generating error file.")
        
        df_invalid = df_aggregated[invalid_groups_mask].reset_index()
        df_invalid = df_invalid.rename(columns=rename_dict)
        df_invalid.to_parquet(to_save_folder / f"{date_str}_errores_month.parquet", engine="pyarrow", compression="snappy")

print("\nProcess Finished.")

Starting 2509...

Process Finished.


Hubieron problemas con 2509 y con 2604 descubrí que es por los cambios de hora

In [16]:
import pandas as pd
from pathlib import Path

# Rutas a los archivos de septiembre (ajusta si usas 2609 o 2509)
archivo_horario = Path(r"E:\ProyectoAnalisisElectrico\MedidasValorizadas\2509\2509_medidas_horarias.parquet")
archivo_mensual = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales\2509\2509_mean_month.parquet")

# 1. Elige cuál archivo quieres abrir (descomenta el que necesites)
# df = pd.read_parquet(archivo_horario)
df = pd.read_parquet(archivo_mensual)

# 2. Ver las columnas y tipos de datos
print("--- ESTRUCTURA DEL ARCHIVO ---")
df.info()

print("\n--- PRIMERAS 5 FILAS ---")
display(df.head())

# 3. Ver una clave específica para revisar la continuidad de las horas
# (Tomamos la primera clave que aparezca en el dataset)
primera_clave = df['clave'].iloc[0]
df_filtro = df[df['clave'] == primera_clave]

print(f"\n--- REVISIÓN DE HORAS PARA LA CLAVE {primera_clave} ---")
# Mostramos las horas ordenadas para ver que no falte ninguna de la 0 a la 23
display(df_filtro[['clave', 'Hora', 'medida_mean', 'CMg[CLP/KWh]_mean']].sort_values('Hora').head(24))

# 4. Verificar que todas las horas tengan la misma cantidad de datos
print("\n--- CONTEO TOTAL DE REGISTROS POR HORA ---")
display(df['Hora'].value_counts().sort_index())

--- ESTRUCTURA DEL ARCHIVO ---
<class 'pandas.DataFrame'>
RangeIndex: 129840 entries, 0 to 129839
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   clave                 129840 non-null  str           
 1   nombre_barra          129840 non-null  str           
 2   tension               129840 non-null  int64         
 3   Zona                  129840 non-null  str           
 4   Razon_Social          129840 non-null  str           
 5   RUT                   129840 non-null  str           
 6   Nombre_Corto          129840 non-null  str           
 7   Hora                  129840 non-null  int32         
 8   Año_Mes               129840 non-null  datetime64[us]
 9   tipo                  129840 non-null  str           
 10  medida_mean           129840 non-null  float64       
 11  medida_std            129840 non-null  float64       
 12  medida_count          129840 non-null 

,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,Hora,Año_Mes,tipo,medida_mean,medida_std,medida_count,CMg[CLP/KWh]_mean,CMg[CLP/KWh]_std,CMg[CLP/KWh]_count,valorizado_CLP_mean,valorizado_CLP_std,valorizado_CLP_count
0,01103243000RC,CALAMA,23,Norte Distribución,Enel Generación Chile S.A.,91.081.000-6,ENEL_GENERACION,0,2025-09-01,L_D,-42.029,44.135048,30,58.950740,8.742176,30,-2534.336159,2882.248945,30
1,01103243000RC,CALAMA,23,Norte Distribución,Enel Generación Chile S.A.,91.081.000-6,ENEL_GENERACION,1,2025-09-01,L_D,-40.826,41.675729,30,59.854353,9.763779,30,-2500.753440,2826.544621,30
2,01103243000RC,CALAMA,23,Norte Distribución,Enel Generación Chile S.A.,91.081.000-6,ENEL_GENERACION,2,2025-09-01,L_D,-39.860,40.474980,30,61.913992,7.690051,30,-2464.173420,2562.869817,30
3,01103243000RC,CALAMA,23,Norte Distribución,Enel Generación Chile S.A.,91.081.000-6,ENEL_GENERACION,3,2025-09-01,L_D,-40.670,42.491916,30,61.701598,8.651902,30,-2527.671967,2744.045522,30
4,01103243000RC,CALAMA,23,Norte Distribución,Enel Generación Chile S.A.,91.081.000-6,ENEL_GENERACION,4,2025-09-01,L_D,-40.704,42.469159,30,62.120274,9.007903,30,-2472.217836,2529.392458,30



--- REVISIÓN DE HORAS PARA LA CLAVE 01103243000RC ---


,clave,Hora,medida_mean,CMg[CLP/KWh]_mean
0,01103243000RC,0,-42.029,58.950740
1,01103243000RC,1,-40.826,59.854353
2,01103243000RC,2,-39.860,61.913992
3,01103243000RC,3,-40.670,61.701598
4,01103243000RC,4,-40.704,62.120274
5,01103243000RC,5,-40.766,62.128686
6,01103243000RC,6,-40.236,65.505107
7,01103243000RC,7,-40.076,64.700398
8,01103243000RC,8,-44.780,14.560726
9,01103243000RC,9,-131.138,3.223647



--- CONTEO TOTAL DE REGISTROS POR HORA ---


Hora
0     5410
1     5410
2     5410
3     5410
4     5410
5     5410
6     5410
7     5410
8     5410
9     5410
10    5410
11    5410
12    5410
13    5410
14    5410
15    5410
16    5410
17    5410
18    5410
19    5410
20    5410
21    5410
22    5410
23    5410
Name: count, dtype: int64